In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/annotated/checkpoints/pre_cell_4.pickle

trying: ['pd']
me:  0
trying: ['load_customer']
me:  3
trying: ['tpch_parent']
me:  0
trying: ['customer']


me:  3
trying: ['load_nation']
me:  4
trying: ['DATA_ROOT']
me:  0
trying: ['nation']
me:  4
trying: ['lineitem']


me:  1
trying: ['load_lineitem']
me:  1
trying: ['load_orders']
me:  2
trying: ['orders']
me:  2
trying: ['STORAGE_OPTS']
me:  0


Declaring variable pd
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable load_orders
Declaring variable orders
Declaring variable load_customer
Declaring variable customer
Declaring variable load_nation
Declaring variable nation


In [4]:
%%RecordEvent
%%time
### cell 4 ###

# Optimized for cudf: avoid CPU-side Timestamp conversions
osel = (orders.O_ORDERDATE >= '1994-11-01') & (orders.O_ORDERDATE < '1995-02-01')
lsel = lineitem.L_RETURNFLAG == 'R'

# Filter once
forders = orders[osel]
flineitem = lineitem[lsel]

# Chain merges to minimize intermediates
jn3 = (flineitem
       .merge(forders,     left_on='L_ORDERKEY', right_on='O_ORDERKEY')
       .merge(customer,    left_on='O_CUSTKEY',  right_on='C_CUSTKEY')
       .merge(nation,      left_on='C_NATIONKEY',right_on='N_NATIONKEY'))

# Compute TMP and aggregate
jn3['TMP'] = jn3.L_EXTENDEDPRICE * (1.0 - jn3.L_DISCOUNT)
total = (
    jn3
    .groupby(
        ['C_CUSTKEY','C_NAME','C_ACCTBAL','C_PHONE','N_NAME','C_ADDRESS','C_COMMENT'],
        as_index=False,
        sort=False
    )['TMP']
    .sum()
    .sort_values('TMP', ascending=False)
)


CPU times: user 95.9 ms, sys: 56 ms, total: 152 ms
Wall time: 162 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q10/rewritten/o4_mini_high/checkpoints/post_cell_4_try_4.pickle

migration speed (bps): 799079065.4861794
---------------------------
variables to migrate:
forders 48
lsel 48
tpch_parent 54
osel 48
STORAGE_OPTS 64
jn3 48
load_lineitem 144
total 48
DATA_ROOT 80
lineitem 48
pd 72
orders 48
flineitem 48
customer 48
load_orders 144
load_nation 144
nation 48
load_customer 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q10/rewritten/o4_mini_high/checkpoints/post_cell_4_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'orders']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['customer', 'lineitem', 'orders']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['forders', 'jn3', 'lsel', 'osel', 'flineitem', 'total']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q10/opt_cell_exec_info_4_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[4], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/annotated/checkpoints/post_cell_4.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['total']
me:  5
trying: ['load_lineitem']
me:  1
trying: ['jn1']
me:  5
trying: ['lineitem']


me:  1
trying: ['lsel']
me:  5
trying: ['date2']
me:  5
trying: ['flineitem']


me:  5
trying: ['DATA_ROOT']
me:  0
trying: ['osel']
me:  5
trying: ['pd']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['customer']
me:  3
trying: ['jn3']
me:  5
trying: ['load_customer']
me:  3
trying: ['load_nation']
me:  4
trying: ['nation']
me:  4
trying: ['orig_output']
me:  6
trying: ['gb']
me:  5
trying: ['jn2']
me:  5
trying: ['orders']


me:  2
trying: ['load_orders']
me:  2
trying: ['date1']
me:  5
trying: ['forders']
me:  5


Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable tpch_parent
Declaring variable STORAGE_OPTS
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable customer
Declaring variable load_customer
Declaring variable load_nation
Declaring variable nation
Declaring variable total
Declaring variable jn1
Declaring variable lsel
Declaring variable date2
Declaring variable flineitem
Declaring variable osel
Declaring variable jn3
Declaring variable gb
Declaring variable jn2
Declaring variable date1
Declaring variable forders
Declaring variable orig_output
